# LoTTE IR Evaluation Notebook

This notebook evaluates the retrieval system using **all qrels queries** from the LoTTE dataset.

**Before running:**
1. Preprocessing, indexing, and retrieval services are running
2. Offline pipeline has built indexes for `lotte/lifestyle/dev/forum`

Set `QUERY_LIMIT = None` for final submission (all qrels). Use `100` for quick local testing.

In [ ]:
import ir_datasets
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.run_evaluation import run_evaluation
from shared.config import DEFAULT_IR_DATASET
from shared.schemas import RetrievalModel

DATASET = DEFAULT_IR_DATASET
QUERY_LIMIT = None  # None = all qrels queries (2076). Use 100 for quick testing.
TOP_K = 10

In [ ]:
dataset = ir_datasets.load(DATASET)
qrels_count = dataset.qrels_count()
queries_count = dataset.queries_count()
docs_count = dataset.docs_count()

print('=' * 70)
print(f'Dataset: {DATASET}')
print(f'Documents: {docs_count:,}')
print(f'Queries: {queries_count:,}')
print(f'Total queries in qrels file: {qrels_count:,}')
print(f'Queries used in this notebook run: {QUERY_LIMIT or qrels_count:,}')
print('=' * 70)

In [ ]:
MODELS = [
    RetrievalModel.TF_IDF,
    RetrievalModel.BM25,
    RetrievalModel.EMBEDDING,
    RetrievalModel.HYBRID_SERIAL,
    RetrievalModel.HYBRID_PARALLEL,
    RetrievalModel.HYBRID_BRANCHING,
]

report = run_evaluation(
    dataset_name=DATASET,
    models=MODELS,
    top_k=TOP_K,
    query_limit=QUERY_LIMIT,
    output_path=Path('data/evaluation/notebook_report.json'),
)
report

In [ ]:
import pandas as pd

df = pd.DataFrame(report['metrics']).T
df.index.name = 'model'
print(f"Evaluated queries: {report['evaluated_queries']} / {report['total_qrels_queries']}")
df